# Functional Visual Field (FVF)
Determine each subject's **functional visual field** - the radius around fixation from which a target is
detected and selected for foveation. This is the input to an inspection-conditioned d' denominator
(`CODE_REVIEW.md` T1), which would replace "every non-target icon" with "items plausibly inspected".
<br><br>
<i>TLDR:</i> _fill in once the numbers below are read_ - the pooled value, the per-subject spread, and
whether the two estimators agree with each other and with `ON_TARGET_THRESHOLD_DVA`.

### Why not the obvious estimator
`P(identified | min eccentricity from any fixation)` is **circular**: marking a target requires foveating
it, so an identification counts as a hit only when gaze is within `ON_TARGET_THRESHOLD_DVA`. Every hit
therefore has minimum eccentricity below that threshold *by construction*, and the curve degenerates into a
step function at the on-target threshold. Two non-circular estimators are compared instead.

**A - foveation falloff.** `P(target ever foveated | minimum distance from any NON-on-target fixation)`.
The outcome is foveation, not identification, which removes the circularity and separates the two
constructs: this curve measures detection-and-selection, while `P(not identified | foveated)` is the LWS
rate. The curve's *level* at small distances carries the LWS floor; its *falloff* carries the field size.

**B - saccade-launch distance.** For each foveated target, the distance from the fixation immediately
preceding its first on-target fixation - the subject selected the target from there.

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.read_data import read_data
from analysis.helpers.fvf import (
    estimate_fvf, estimate_by_foveation_falloff, estimate_by_launch_distance,
)

pio.renderers.default = 'notebook'      # 'notebook' or 'browser'

# compare against the a-priori constant rather than hard-coding a number
ON_TARGET = cnfg.ON_TARGET_THRESHOLD_DVA
PERCENTILES = [0.5, 0.75, 0.85, 0.9, 0.95, 0.99]

### Read data

In [ ]:
loaded_data = read_data(cnfg.OUTPUT_PATH)
fixations = loaded_data.fixations
idents = loaded_data.identifications
del loaded_data

## (A) Foveation falloff

For every (subject, trial, target): the outcome is whether the target was **ever foveated**; the predictor
is the closest the subject came to it **without** foveating it. The field edge is where the curve drops to
half its near-fovea level.

In [ ]:
falloff_by_subject, falloff_pooled, curve = estimate_by_foveation_falloff(fixations, ON_TARGET)

print(f'Pooled FVF (foveation falloff): {falloff_pooled:.2f} DVA')
print(f"Near-fovea foveation rate: {curve['rate'].iloc[0]:.3f} "
      f'(below 1 means targets approached closely but never foveated)')
curve

In [ ]:
falloff_by_subject.describe(PERCENTILES).to_frame('foveation_falloff_dva').T

## (B) Saccade-launch distance

For every target that *was* foveated: how far away was the fixation the subject launched from? A high
percentile of that distribution is the practical edge of the field.

In [ ]:
launch_by_subject, launch_pooled, launches = estimate_by_launch_distance(fixations, ON_TARGET)

print(f'Pooled FVF (launch distance, 95th pct): {launch_pooled:.2f} DVA   (n = {len(launches):,})')

launch_summary = (
    pd.concat([
        launches['launch_dva'].describe(PERCENTILES).rename('all'),
        launches.groupby('subject', observed=True)['launch_dva'].describe(PERCENTILES).T,
    ], axis=1)
).T
launch_summary

## Comparison

The two estimators and the a-priori `ON_TARGET_THRESHOLD_DVA`, side by side. Substantial disagreement -
between the estimators, or between either and the threshold - is itself a finding and belongs in the TLDR.

In [ ]:
comparison = estimate_fvf(fixations, ON_TARGET).assign(
    falloff_minus_launch=lambda df: df['foveation_falloff'] - df['launch_distance'],
    falloff_over_threshold=lambda df: df['foveation_falloff'] / df['on_target_threshold'],
)
comparison.round(2)

In [ ]:
pooled = comparison.loc['all']
print(f"pooled foveation-falloff : {pooled['foveation_falloff']:.2f} DVA")
print(f"pooled launch-distance   : {pooled['launch_distance']:.2f} DVA")
print(f"on-target threshold      : {pooled['on_target_threshold']:.2f} DVA")
print(f"estimators differ by     : {abs(pooled['foveation_falloff'] - pooled['launch_distance']):.2f} DVA")
print(f"field / on-target ratio  : {pooled['foveation_falloff'] / pooled['on_target_threshold']:.1f}x")

### Visualize

Row 1 pools all subjects; row 2 shows each subject separately, so heterogeneity is visible before a single
pooled number is adopted.

In [ ]:
fig = make_subplots(rows=2, cols=2, shared_xaxes=True,
                    column_titles=['Foveation Falloff (A)', 'Launch Distance (B)'])

fig.add_trace(row=1, col=1, trace=go.Scatter(
    x=curve['centre'], y=curve['rate'], mode='lines+markers', name='P(foveated)',
    line=dict(color=cnfg.get_discrete_color('all'), width=3),
))
fig.add_vline(x=falloff_pooled, row=1, col=1, line=dict(dash='dash', width=2))

fig.add_trace(row=1, col=2, trace=go.Violin(
    x=launches['launch_dva'], name='All Subjects', orientation='h', side='positive',
    spanmode='hard', width=1.75, points='all', pointpos=-0.5,
    meanline=dict(visible=True), box=dict(visible=False),
    line=dict(color=cnfg.get_discrete_color('all')),
))

for subj, grp in launches.groupby('subject', observed=True):
    colour = cnfg.get_discrete_color(int(subj), loop=True)
    sub_curve = estimate_by_foveation_falloff(fixations[fixations['subject'] == subj], ON_TARGET)[2]
    fig.add_trace(row=2, col=1, trace=go.Scatter(
        x=sub_curve['centre'], y=sub_curve['rate'], mode='lines', name=f'S{subj}',
        line=dict(color=colour, width=1.5), opacity=0.7, showlegend=False,
    ))
    fig.add_trace(row=2, col=2, trace=go.Violin(
        x=grp['launch_dva'], name=f'S{subj}', orientation='h', side='positive',
        spanmode='hard', width=1.75, points=False, meanline=dict(visible=True),
        box=dict(visible=False), line=dict(color=colour), showlegend=False,
    ))

fig.update_xaxes(title=dict(text='Distance from Fixation (DVA)', font=cnfg.AXIS_LABEL_FONT),
                 tickfont=cnfg.AXIS_TICK_FONT, gridcolor=cnfg.GRID_LINE_COLOR, row=2)
fig.update_yaxes(showticklabels=False, gridcolor=cnfg.GRID_LINE_COLOR)
fig.update_layout(width=1300, height=700, paper_bgcolor='rgba(0, 0, 0, 0)',
                  title=dict(text='Functional Visual Field', font=cnfg.TITLE_FONT))
fig.show()

### Landing the value

Nothing consumes the FVF yet - the d' denominator is a separate open task (`CODE_REVIEW.md` T1) and the
coverage/mapping work is deferred. **No constant is transcribed into `config.py` or `funnel_config.py`**: a
hand-copied per-subject dict would go stale the moment the pipeline is re-run. Callers should call
`estimate_fvf()` and pass a radius explicitly; this notebook justifies the value they choose.